# Interruptible MLE agent: one run on Colab

Starts one run, or resumes it if it was cut off. The run lives on the VM's local disk and is copied
to Google Drive every two minutes, so after a disconnect you **reconnect and run all cells again**:
the run continues from the last copy. See `docs/cloud-runs.md`.

**Before the first run**
- Secrets (key icon, left bar), with notebook access on: `GITHUB_TOKEN` (fine-grained, read-only
  *Contents* on this repo), and `OPENROUTER_API_KEY` and/or `GEMINI_API_KEY`.
- *Runtime → Change runtime type*: CPU.
- A real task (`SMOKE = False`) needs role 3's evaluator and the prepared data. Until then, keep `SMOKE = True`.

**While it runs**: keep this tab open and active. The free tier disconnects after about 90 idle
minutes and ends sessions after about 12 hours. Docker is not available, so scripts run with
`--sandbox subprocess` (no isolation); the report states this.

In [ ]:
# Settings: the only cell to edit.
CONDITION = "configs/conditions/c1_autonomous.yaml"
TASK = "spooky-author-identification"
SEED = 0
PROVIDER = "openrouter"   # openrouter (experiments) | gemini (cheap testing)
BRANCH = "main"           # "nabil-coreagent" until PR #2 is merged
SMOKE = True              # True: toy task, no evaluator or Kaggle data needed. False: the real task
SMOKE_NODES = 20
DRY_RUN = False           # with SMOKE: canned replies, no API key needed
DRIVE_RUNS = "/content/drive/MyDrive/imle/runs"  # where runs are copied and restored from

In [ ]:
import os
import subprocess
import sys

REPO = "github.com/nabilrakaiza/interruptible-mle-agent.git"


def git(*args, token=""):
    """Run git without ever printing the token, even when it fails."""
    result = subprocess.run(["git", *args], capture_output=True, text=True, check=False)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError("git failed: " + message)


def get_code(dest, token):
    """Clone the private repo (or update it), then remove the token from .git/config."""
    url = f"https://x-access-token:{token}@{REPO}"
    if not os.path.exists(dest):
        git("clone", "-q", "--branch", BRANCH, url, dest, token=token)
    else:
        git("-C", dest, "fetch", "-q", url, BRANCH, token=token)
        git("-C", dest, "checkout", "-q", "FETCH_HEAD", token=token)
    git("-C", dest, "remote", "set-url", "origin", f"https://{REPO}")


from google.colab import drive, userdata

drive.mount("/content/drive")
for name in ["OPENROUTER_API_KEY", "GEMINI_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:  # noqa: BLE001 - the platforms raise their own types; not every key is needed
        print(f"{name} is not in Colab secrets; fine unless PROVIDER needs it")

CODE = "/content/imle"
get_code(CODE, userdata.get("GITHUB_TOKEN"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[sandbox,analysis]"], cwd=CODE, check=True)

In [ ]:
RUNS_ROOT = "/content/runs"
SYNC = ["--sync-to", DRIVE_RUNS]

cmd = [sys.executable, "scripts/cloud_run.py", "--seed", str(SEED), "--runs-root", RUNS_ROOT]
cmd += SYNC
if SMOKE:
    cmd += ["--smoke", "--nodes", str(SMOKE_NODES)] + (["--dry-run"] if DRY_RUN else ["--provider", PROVIDER])
else:
    cmd += ["--config", CONDITION, "--task", TASK, "--provider", PROVIDER]
print(" ".join(cmd))
# Stream the run's output into this cell as it happens.
process = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
print("exit code", process.wait())

## Look at the result

The trajectory figure and timeline are written into the run folder (and copied to Drive with it).

In [ ]:
import glob

for run in sorted(glob.glob(f"{RUNS_ROOT}/*")):
    subprocess.run([sys.executable, "scripts/trajectory.py", run], cwd=CODE, check=False, capture_output=True)
    print(run, "->", f"{run}/trajectory/")